In [1]:
import ee, pandas as pd

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()
    
print("Earth Engine is ready!")
print("EE Version:", ee.__version__)

Earth Engine is ready!
EE Version: 1.6.14


# 1. Define Area of Interest (AOI)

Here we use US Census TIGER/2018 county boundaries. STATEFP = 37 corresponds to North Carolina and then we select Mecklenburg County.

In [2]:
AOI = (ee.FeatureCollection('TIGER/2018/Counties')
        .filter(ee.Filter.eq('NAME', 'Mecklenburg'))
        .filter(ee.Filter.eq('STATEFP', '37')))


# 2. Define the Temporal Study Window

Defines the full temporal range of interest for all datasets.

In [3]:
DATE_START = '1990-01-01'
DATE_END   = '2024-12-31'

# 3. Image Collection Inventory & Metadata Inspection
Inspects:
- Total image count within AOI
- Temporal coverage (first/last year)
- Available years
- Band configurations

In [5]:
def inspect_collection_inventory(collection_id, aoi = AOI):
    col = ee.ImageCollection(collection_id).filterBounds(aoi)

    def add_props(img):
        # Some datasets may not have time_start → guard against nulls
        time = ee.Date(img.get('system:time_start'))
        yr = ee.Algorithms.If(
            img.get('system:time_start'),
            time.get('year'),
            None
        )
        bands = img.bandNames().join(',')
        return img.set({'year': yr, 'bands_str': bands})

    col = col.map(add_props)

    years = ee.List(col.aggregate_array('year')) \
                .removeAll([None]) \
                .distinct() \
                .sort()

    band_sets = ee.List(col.aggregate_array('bands_str')).distinct()

    years_py = years.getInfo()

    return {
        'Collection': collection_id,
        'Image Count': col.size().getInfo(),
        'First Year': min(years_py) if years_py else None,
        'Last Year': max(years_py) if years_py else None,
        'Years Available': years_py,
        'Band Sets': band_sets.getInfo()
    }

report_results = []

In [6]:
report_results.append(inspect_collection_inventory('USDA/NAIP/DOQQ'))                     # NAIP
report_results.append(inspect_collection_inventory('LANDSAT/LT05/C02/T1_L2'))             # Landsat 5 TM
report_results.append(inspect_collection_inventory('LANDSAT/LE07/C02/T1_L2'))             # Landsat 7 ETM+
report_results.append(inspect_collection_inventory('LANDSAT/LC08/C02/T1_L2'))             # Landsat 8 OLI/TIRS
report_results.append(inspect_collection_inventory('LANDSAT/LC09/C02/T1_L2'))             # Landsat 9 OLI-2/TIRS-2
report_results.append(inspect_collection_inventory('COPERNICUS/S2_SR_HARMONIZED'))        # Sentinel-2 SR (harmonized)
report_results.append(inspect_collection_inventory('MODIS/061/MOD11A2'))                  # MODIS LST
report_results.append(inspect_collection_inventory('MODIS/061/MOD13Q1'))                  # MODIS NDVI/EVI
report_results.append(inspect_collection_inventory('USGS/NLCD_RELEASES/2019_REL/NLCD'))   # NLCD
report_results.append(inspect_collection_inventory('ESA/WorldCover/v200'))                # ESA WorldCover

df_collections = pd.DataFrame(report_results); df_collections

,Collection,Image Count,First Year,Last Year,Years Available,Band Sets
0,USDA/NAIP/DOQQ,744,2004,2023,"[2004, 2005, 2006, 2008, 2009, 2010, 2011, 201...","[R,G,B,N, R,G,B]"
1,LANDSAT/LT05/C02/T1_L2,1337,1984,2011,"[1984, 1985, 1986, 1987, 1988, 1989, 1990, 199...","[SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_..."
2,LANDSAT/LE07/C02/T1_L2,1391,1999,2023,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...","[SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_..."
3,LANDSAT/LC08/C02/T1_L2,740,2013,2026,"[2013, 2014, 2015, 2016, 2017, 2018, 2019, 202...","[SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B6,SR_B7,SR_..."
4,LANDSAT/LC09/C02/T1_L2,253,2021,2026,"[2021, 2022, 2023, 2024, 2025, 2026]","[SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B6,SR_B7,SR_..."
5,COPERNICUS/S2_SR_HARMONIZED,5282,2015,2026,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...","[B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12,AOT,WV..."
6,MODIS/061/MOD11A2,1194,2000,2026,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...","[LST_Day_1km,QC_Day,Day_view_time,Day_view_ang..."
7,MODIS/061/MOD13Q1,597,2000,2026,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...","[NDVI,EVI,DetailedQA,sur_refl_b01,sur_refl_b02..."
8,USGS/NLCD_RELEASES/2019_REL/NLCD,8,2001,2019,"[2001, 2004, 2006, 2008, 2011, 2013, 2016, 2019]","[landcover,impervious,impervious_descriptor]"
9,ESA/WorldCover/v200,1,2021,2021,[2021],[Map]


# 4. Specific Collection's Year-by-Year Coverage and Resolution Audit

Evaluates:
- Image availability per year
- Band configuration stability
- Finest spatial resolution by NAIP era

**USDA/NAIP/DOQQ**

In [4]:
COLLECTION_ID = 'USDA/NAIP/DOQQ'
first_year = 2004
last_year = 2023

In [5]:
def inspect_naip_by_year_bounded(collection_id, start_year, end_year, aoi=AOI):
    col = ee.ImageCollection(collection_id).filterBounds(aoi)

    years = ee.List.sequence(start_year, end_year)

    def naip_resolution(year):
        year = ee.Number(year)
        return ee.Algorithms.If(
            year.lte(2003), 2,
            ee.Algorithms.If(year.lt(2018), 1, 0.6)
        )

    def inspect_year(y):
        y = ee.Number(y)
        yearly = col.filter(ee.Filter.calendarRange(y, y, 'year'))
        count = yearly.size()

        # Safe conditional logic
        band_sig = ee.Algorithms.If(
            count.gt(0),
            ee.Image(yearly.first()).bandNames().join(','),
            'NO_DATA'
        )

        return ee.Feature(None, {
            'year': y,
            'image_count': count,
            'band_signature': band_sig,
            'finest_resolution_m': naip_resolution(y),
            'platform': 'NAIP'
        })

    return ee.FeatureCollection(years.map(inspect_year))

fc_naip_bounded = inspect_naip_by_year_bounded(COLLECTION_ID, first_year, last_year)

naip_stats_bounded = {
    'year': fc_naip_bounded.aggregate_array('year').getInfo(),
    'image_count': fc_naip_bounded.aggregate_array('image_count').getInfo(),
    'band_signature': fc_naip_bounded.aggregate_array('band_signature').getInfo(),
    'finest_resolution_m': fc_naip_bounded.aggregate_array('finest_resolution_m').getInfo(),
    'platform': fc_naip_bounded.aggregate_array('platform').getInfo(),
}

df_naip_bounded = (pd.DataFrame(naip_stats_bounded).sort_values('year').reset_index(drop=True)); df_naip_bounded

,year,image_count,band_signature,finest_resolution_m,platform
0,2004,55,"R,G,B",1.0,NAIP
1,2005,64,"R,G,B",1.0,NAIP
2,2006,56,"R,G,B",1.0,NAIP
3,2007,0,NO_DATA,1.0,NAIP
4,2008,56,"R,G,B",1.0,NAIP
5,2009,64,"R,G,B,N",1.0,NAIP
6,2010,56,"R,G,B,N",1.0,NAIP
7,2011,8,"R,G,B,N",1.0,NAIP
8,2012,56,"R,G,B,N",1.0,NAIP
9,2013,8,"R,G,B,N",1.0,NAIP


**LANDSAT/LT05/C02/T1_L2**

In [6]:
COLLECTION_ID = 'LANDSAT/LT05/C02/T1_L2'
first_year = 1984
last_year = 2011

In [9]:
# Function to inspect Landsat 5 by year
def inspect_landsat5_by_year(collection_id, start_year, end_year, aoi=AOI):
    col = ee.ImageCollection(collection_id).filterBounds(aoi)

    years = ee.List.sequence(start_year, end_year)

    def inspect_year(y):
        y = ee.Number(y)
        yearly = col.filter(ee.Filter.calendarRange(y, y, 'year'))

        count = yearly.size()
        
        # band signature
        band_sig = ee.Algorithms.If(
            count.gt(0),
            ee.Image(yearly.first()).bandNames().join(','),
            'NO_DATA'
        )
        
        # resolution (nominal 30m for all bands in this Earth Engine product)
        # If no images, we still report 30m as nominal
        res_nominal = ee.Number(30)

        return ee.Feature(None, {
            'year': y,
            'image_count': count,
            'band_signature': band_sig,
            'nominal_resolution_m': res_nominal
        })

    return ee.FeatureCollection(years.map(inspect_year))

fc_l5_bounded = inspect_landsat5_by_year(COLLECTION_ID, first_year, last_year)

l5_stats_bounded = {
    'year': fc_l5_bounded.aggregate_array('year').getInfo(),
    'image_count': fc_l5_bounded.aggregate_array('image_count').getInfo(),
    'band_signature': fc_l5_bounded.aggregate_array('band_signature').getInfo(),
    'nominal_resolution_m': fc_l5_bounded.aggregate_array('nominal_resolution_m').getInfo(),
}

df_l5_bounded = (pd.DataFrame(l5_stats_bounded)
                  .sort_values('year')
                  .reset_index(drop=True))
df_l5_bounded

,year,image_count,band_signature,nominal_resolution_m
0,1984,28,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
1,1985,26,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
2,1986,56,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
3,1987,39,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
4,1988,55,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
5,1989,52,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
6,1990,45,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
7,1991,53,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
8,1992,50,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30
9,1993,50,"SR_B1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,SR_ATMOS_O...",30


In [4]:
!jupyter nbconvert --to html --no-input metadata_collection.ipynb --output ../../../output/Notebook_Outputs/RS/metadata_collection.html

[NbConvertApp] Converting notebook metadata_collection.ipynb to html
[NbConvertApp] Writing 591468 bytes to ..\..\..\output\Notebook_Outputs\RS\metadata_collection.html
